In [1]:
import os
import pandas as pd

data_path = "../data/archive"
output_path = "../data/output"

os.makedirs(output_path, exist_ok=True)

In [2]:
kjv = pd.read_csv(os.path.join(data_path, "t_kjv.csv"))
book_key = pd.read_csv(os.path.join(data_path, "key_english.csv"))
genre_key = pd.read_csv(os.path.join(data_path, "key_genre_english.csv"))

In [3]:
book_stats = (
    kjv
    .groupby('b')
    .agg(
        total_chapters=('c', 'nunique'),
        total_verses=('v', 'count'),
        total_characters=('t', lambda x: x.str.len().sum()),
        total_words=('t', lambda x: x.str.split().str.len().sum())
    )
    .reset_index()
)

book_key = book_key.rename(columns={
    'n': 'book_name',
    'g': 'genre_id'
})

genre_key = genre_key.rename(columns={
    'n': 'genre_name',
    'g': 'genre_id'
})

lib = (
    book_stats
    .merge(book_key[['b','book_name','genre_id']], on='b', how='left')
    .merge(genre_key[['genre_id','genre_name']], on='genre_id', how='left')
)

lib = lib.rename(columns={'b': 'book_number'})

lib['testament'] = lib['book_number'].apply(
    lambda x: 'Old Testament' if x <= 39 else 'New Testament'
)

lib['avg_chars_per_verse'] = lib['total_characters'] / lib['total_verses']
lib['avg_words_per_verse'] = lib['total_words'] / lib['total_verses']

lib.head()

,book_number,total_chapters,total_verses,total_characters,total_words,book_name,genre_id,genre_name,testament,avg_chars_per_verse,avg_words_per_verse
0,1,50,1533,195262,38265,Genesis,1,Law,Old Testament,127.372472,24.960861
1,2,40,1213,168113,32684,Exodus,1,Law,Old Testament,138.592745,26.944765
2,3,27,859,125992,24543,Leviticus,1,Law,Old Testament,146.672875,28.571595
3,4,36,1288,174297,32895,Numbers,1,Law,Old Testament,135.323758,25.539596
4,5,34,959,145443,28351,Deuteronomy,1,Law,Old Testament,151.661105,29.563087


In [4]:
delimiter = "|"
print("Delimiter:", delimiter)

num_obs = lib.shape[0]
print("Number of observations:", num_obs)

print("All features:")
for col in lib.columns:
    print("-", col)

model_features = [
    "genre_name",
    "testament",
    "total_words",
    "avg_words_per_verse",
    "total_characters"
]

print("\nFeatures usable for model summarization:")
for f in model_features:
    print("-", f)

avg_doc_length_chars = lib["total_characters"].mean()
print("Average length of each document (in characters):", round(avg_doc_length_chars, 2))

Delimiter: |
Number of observations: 66
All features:
- book_number
- total_chapters
- total_verses
- total_characters
- total_words
- book_name
- genre_id
- genre_name
- testament
- avg_chars_per_verse
- avg_words_per_verse

Features usable for model summarization:
- genre_name
- testament
- total_words
- avg_words_per_verse
- total_characters
Average length of each document (in characters): 62217.39


In [5]:
lib.to_csv(os.path.join(output_path, "LIB.csv"), index=False, sep="|")

print("LIB table saved.")
print("Shape:", lib.shape)

LIB table saved.
Shape: (66, 11)
